# Laboratorio: aproximación lineal y mínimos cuadrados con QR
**Métodos Numéricos — Facultad de Economía, Universidad del Pacífico**  
**Periodo:** 2026-2 · **Duración sugerida:** 110 minutos · **Python 3 / Google Colab**

**Nombre(s):** …  
**Fecha:** …

Entregamos un conjunto de datos y un catálogo de funciones. Usted seleccionará funciones del catálogo, las evaluará en los datos, construirá las ecuaciones normales y resolverá el problema de aproximación mediante QR.

Al terminar podrá:
1. reconocer un modelo lineal en sus coeficientes, aunque sus funciones base no sean rectas;
2. construir la matriz de diseño a partir de evaluaciones;
3. formar e interpretar las ecuaciones normales;
4. calcular el ajuste mediante QR y sustitución hacia atrás;
5. verificar la ortogonalidad del residuo y comparar modelos.

**Colab:** suba este archivo `.ipynb` mediante **Archivo → Subir cuaderno**, guarde una copia y ejecute las celdas en orden. Los datos están incluidos: no se requieren descargas ni archivos externos. Complete las respuestas y los experimentos indicados. Las celdas resueltas sirven como guía; no sustituyen las justificaciones.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
np.set_printoptions(precision=6, suppress=True, linewidth=120)

## 1. Datos entregados: ventas mensuales
Una empresa registra ventas mensuales (en miles de soles) durante 24 meses. El índice temporal es $t=1,\ldots,24$. **Son datos simulados para fines docentes; no representan una empresa real.** Se observan crecimiento y oscilaciones; buscamos una función que aproxime las observaciones, sin exigir que pase por todos los puntos.

La escala temporal de trabajo será $x=t/12$ (años desde el origen del índice). No cambie esta escala durante la comparación. Las funciones logarítmicas se evalúan únicamente para $x>0$.

**Actividad 1 (10 min).** Observe la nube de datos. Describa dos patrones y proponga qué funciones del catálogo podrían representarlos. Explique por qué no se trata de interpolación.

In [ ]:
t = np.arange(1, 25, dtype=float)
y = np.array([43.8, 46.7, 49.1, 48.5, 48.0, 45.2, 42.7, 40.6, 39.8, 42.3, 45.6, 50.1, 55.2, 59.0, 60.8, 61.5, 59.4, 57.1, 54.3, 52.8, 53.2, 55.7, 59.8, 65.3], dtype=float)
x = t / 12.0
datos = pd.DataFrame({"mes": t.astype(int), "x_anios": x, "ventas_miles_S": y})
display(datos)
plt.figure(figsize=(9, 4))
plt.scatter(t, y, color="#003b70")
plt.xlabel("Mes")
plt.ylabel("Ventas (miles de soles)")
plt.title("Datos entregados")
plt.grid(alpha=.25)
plt.show()

**Respuesta 1.** Escriba aquí su interpretación de los datos y su propuesta inicial.

## 2. Catálogo y función de aproximación
Seleccione un subconjunto ordenado de funciones. Cada función seleccionada corresponde a una columna y a un coeficiente.

| Clave | Función $\phi(x)$ | Posible componente |
|---|---|---|
| `constante` | $1$ | nivel |
| `lineal` | $x$ | tendencia lineal |
| `cuadratica` | $x^2$ | curvatura |
| `cubica` | $x^3$ | curvatura adicional |
| `logaritmo` | $\log x$ | crecimiento de pendiente decreciente |
| `raiz` | $\sqrt{x}$ | crecimiento de pendiente decreciente |
| `seno` | $\sin(2\pi x)$ | ciclo anual |
| `coseno` | $\cos(2\pi x)$ | ciclo anual con otra fase |

Para $p$ funciones seleccionadas definimos
$$\widehat f(x)=\sum_{j=1}^{p} c_j\phi_j(x).$$
“Lineal” se refiere a los coeficientes $c_j$, no necesariamente a la variable $x$. Las funciones y sus parámetros internos están fijados: aquí no estimamos una frecuencia ni un exponente.

**Actividad 2 (15 min).** Usaremos primero `constante`, `lineal`, `seno`, `coseno`. Escriba $\widehat f(x)$ y la expresión de $\widehat f(t/12)$ en términos del mes. ¿Por qué seno y coseno se usan juntos? ¿Cuántos coeficientes se estiman?

In [ ]:
catalogo = {
    "constante": lambda z: np.ones_like(z, dtype=float),
    "lineal": lambda z: z,
    "cuadratica": lambda z: z**2,
    "cubica": lambda z: z**3,
    "logaritmo": lambda z: np.log(z),
    "raiz": lambda z: np.sqrt(z),
    "seno": lambda z: np.sin(2*np.pi*z),
    "coseno": lambda z: np.cos(2*np.pi*z),
}

def matriz_diseno(z, seleccion):
    z = np.asarray(z, dtype=float)
    if z.ndim != 1 or not np.all(np.isfinite(z)):
        raise ValueError("Se requiere un vector finito de abscisas.")
    if not seleccion or len(set(seleccion)) != len(seleccion):
        raise ValueError("Seleccione funciones distintas, sin repetir claves.")
    if any(k not in catalogo for k in seleccion):
        raise ValueError("Hay una clave que no pertenece al catálogo.")
    if "logaritmo" in seleccion and np.any(z <= 0):
        raise ValueError("El logaritmo requiere x > 0.")
    if "raiz" in seleccion and np.any(z < 0):
        raise ValueError("La raíz requiere x >= 0.")
    return np.column_stack([catalogo[k](z) for k in seleccion])

seleccion = ["constante", "lineal", "seno", "coseno"]
A = matriz_diseno(x, seleccion)
m, p = A.shape
print(f"A tiene {m} filas y {p} columnas; y tiene {len(y)} componentes.")
display(pd.DataFrame(A, columns=seleccion, index=t.astype(int)).rename_axis("mes"))
print("Rango columna:", np.linalg.matrix_rank(A))

**Respuesta 2.** Escriba el modelo y justifique la selección.

**Actividad 3 (10 min).** Verifique a mano las filas correspondientes a los meses 3 y 12, incluyendo los valores trigonométricos. Identifique el significado de $A_{ij}=\phi_j(x_i)$ y explique por qué el sistema $Ac=y$ tiene más ecuaciones que incógnitas.

**Respuesta 3.** Anote sus cálculos y compare con la tabla.

## 3. Evaluación de las bases y ecuaciones normales
El objetivo es minimizar la suma de cuadrados de los residuos verticales:
$$J(c)=\|y-Ac\|_2^2=\sum_{i=1}^{m}\left(y_i-\sum_{j=1}^{p}c_j\phi_j(x_i)\right)^2.$$
Al derivar respecto de cada coeficiente:
$$\frac{\partial J}{\partial c_k}=-2\sum_{i=1}^{m}\phi_k(x_i)\left(y_i-\sum_{j=1}^{p}c_j\phi_j(x_i)\right)=0.$$
Así se forman las **ecuaciones normales**:
$$Gc=h,\qquad G=A^TA,\quad h=A^Ty,$$
$$G_{kj}=\sum_{i=1}^{m}\phi_k(x_i)\phi_j(x_i),\qquad h_k=\sum_{i=1}^{m}\phi_k(x_i)y_i.$$

**Actividad 4 (15 min).** Calcule con sumatorias $G_{11},G_{12},G_{22},h_1,h_2$ y contraste con el producto matricial. Escriba las cuatro ecuaciones numéricas del modelo inicial (redondee solo para presentarlas). Explique por qué $G$ es simétrica y cuándo es definida positiva.

In [ ]:
G = A.T @ A
h = A.T @ y
display(pd.DataFrame(G, index=seleccion, columns=seleccion))
display(pd.Series(h, index=seleccion, name="A^T y"))
print("Comprobación por sumatorias:")
for k, j in [(0, 0), (0, 1), (1, 1)]:
    valor = np.sum(A[:, k] * A[:, j])
    print(f"G[{k+1},{j+1}] = {valor:.8f}")
for k in [0, 1]:
    print(f"h[{k+1}] = {np.sum(A[:, k]*y):.8f}")
print("Ecuaciones normales (coeficientes en el orden de selección):")
for fila, rhs in zip(G, h):
    print(" + ".join(f"({a:.6f}) c{j+1}" for j, a in enumerate(fila)), "=", f"{rhs:.6f}")

**Respuesta 4.** Escriba las ecuaciones, los cálculos pedidos y la condición de unicidad.

## 4. Resolución mediante QR
Formamos las ecuaciones normales para comprender el problema. Para calcular $c$ factorizamos **la matriz rectangular $A$** mediante QR reducida:
$$A=QR,\qquad Q\in\mathbb R^{m\times p},\quad Q^TQ=I_p,\quad R\in\mathbb R^{p\times p}.$$
Si $A$ tiene rango columna completo, $R$ es triangular superior no singular y el ajuste se obtiene de
$$Rc=Q^Ty.$$
Esto también se deduce de $A^TA=R^TR$ y $A^Ty=R^TQ^Ty$. Se evita formar $A^TA$ como vía de solución: en norma 2, para rango completo, $\kappa_2(A^TA)=\kappa_2(A)^2$.

Usaremos la rutina QR de NumPy y una sustitución hacia atrás explícita. No use inversas ni `lstsq` para obtener el ajuste principal; `lstsq` aparecerá al final únicamente como referencia.

**Actividad 5 (20 min).** Antes de ejecutar, escriba las dimensiones de $Q,R,Q^Ty,c$. Explique por qué $Q^TQ=I_p$ no implica $QQ^T=I_m$. Interprete las comprobaciones de reconstrucción y ortogonalidad.

In [ ]:
def sustitucion_atras(R, b):
    R = np.asarray(R, dtype=float)
    b = np.asarray(b, dtype=float)
    n = len(b)
    if R.shape != (n, n):
        raise ValueError("Dimensiones incompatibles.")
    c = np.zeros(n)
    for i in range(n-1, -1, -1):
        if R[i, i] == 0:
            raise np.linalg.LinAlgError("Pivote nulo.")
        c[i] = (b[i] - R[i, i+1:] @ c[i+1:]) / R[i, i]
    return c

def ajustar_qr(A, y):
    A = np.asarray(A, dtype=float)
    y = np.asarray(y, dtype=float)
    m, p = A.shape
    if y.shape != (m,) or m < p:
        raise ValueError("Se requiere m >= p y un dato por fila.")
    if np.linalg.matrix_rank(A) < p:
        raise np.linalg.LinAlgError("Las columnas no tienen rango completo: revise las bases.")
    Q, R = np.linalg.qr(A, mode="reduced")
    b = Q.T @ y
    c = sustitucion_atras(R, b)
    return c, Q, R, b

c, Q, R, b = ajustar_qr(A, y)
print("Dimensiones:", Q.shape, R.shape, b.shape, c.shape)
print("R =", R)
print("Q^T y =", b)
display(pd.Series(c, index=seleccion, name="Coeficiente QR"))
print("Error relativo de reconstrucción:", np.linalg.norm(A-Q@R)/np.linalg.norm(A))
print("Error de ortogonalidad:", np.linalg.norm(Q.T@Q-np.eye(p)))
assert np.allclose(Q@R, A)
assert np.allclose(Q.T@Q, np.eye(p))
assert np.allclose(R@c, b)

**Respuesta 5.** Explique la reducción triangular y escriba la función ajustada con sus coeficientes.

## 5. Evaluación de la aproximación y verificación
Definimos $\widehat y=Ac$ y $r=y-\widehat y$. El ajuste mínimo cuadrático satisface $A^Tr\approx0$: el residuo es ortogonal a todas las columnas de $A$. No se espera $r=0$.

Reportaremos $\mathrm{SSE}=r^Tr$ y $\mathrm{RMSE}=\sqrt{\mathrm{SSE}/m}$ (error descriptivo de ajuste; no es la estimación de la desviación estándar con divisor $m-p$).

**Actividad 6 (15 min).** Identifique el mes con mayor residuo absoluto y su signo. Explique el significado de $A^Tr\approx0$ y por qué una columna constante hace que los residuos sumen aproximadamente cero. Evalúe la función en el mes 18.5 e indique si es interpolación de la función ajustada dentro del intervalo o extrapolación. Comente si la gráfica de residuos sugiere un patrón pendiente de explicar.

In [ ]:
y_ajustada = A @ c
r = y - y_ajustada
SSE = r @ r
RMSE = np.sqrt(SSE/m)
display(pd.DataFrame({"mes": t.astype(int), "observado": y, "ajustado": y_ajustada, "residuo": r}))
print("SSE:", SSE, " RMSE:", RMSE)
print("A^T r:", A.T @ r)
print("Residuo de las ecuaciones normales:", np.linalg.norm(G@c-h))
print("Suma de residuos:", np.sum(r))
print("Mes con mayor |residuo|:", int(t[np.argmax(np.abs(r))]))
print("Estimación en el mes 18.5:", (matriz_diseno(np.array([18.5/12]), seleccion) @ c)[0])
assert np.linalg.norm(A.T@r) < 1e-9*np.linalg.norm(A)*np.linalg.norm(y)
c_ref = np.linalg.lstsq(A, y, rcond=None)[0]
print("Diferencia respecto de referencia:", np.linalg.norm(c-c_ref))
assert np.allclose(c, c_ref)

meses_finos = np.linspace(1, 24, 400)
curva = matriz_diseno(meses_finos/12, seleccion) @ c
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
ejes[0].scatter(t, y, label="Datos", color="#003b70")
ejes[0].plot(meses_finos, curva, color="#d07900", label="Aproximación QR")
ejes[0].set(xlabel="Mes", ylabel="Ventas (miles de soles)")
ejes[0].legend()
ejes[1].scatter(t, r, color="#003b70")
ejes[1].axhline(0, color="black", linewidth=1)
ejes[1].set(xlabel="Mes", ylabel="Residuo (observado − ajustado)")
for eje in ejes: eje.grid(alpha=.25)
plt.tight_layout()
plt.show()

**Respuesta 6.** Interprete los coeficientes, el mayor residuo, la ortogonalidad y la estimación pedida.

## 6. Experimento: elegir las funciones del catálogo
**Actividad 7 (20 min).** Compare las cuatro selecciones siguientes. Agregue una quinta de su elección con un máximo de seis funciones distintas. Todos los modelos deben usar los mismos 24 datos y resolverse con QR.

1. Recta: $1,x$.
2. Tendencia cuadrática: $1,x,x^2$.
3. Tendencia con ciclo anual: $1,x,\sin(2\pi x),\cos(2\pi x)$.
4. Curvatura con ciclo anual: $1,x,x^2,\sin(2\pi x),\cos(2\pi x)$.

Para su modelo elegido, muestre la matriz de evaluaciones, las ecuaciones normales, los coeficientes QR, la gráfica y $A^Tr$. Puede cambiar `seleccion` en la sección 2 y volver a ejecutar las secciones 2–5.

El error de entrenamiento no aumenta al agregar columnas a un modelo anidado en aritmética exacta. Un menor SSE no basta para justificar mayor complejidad ni garantiza mejores predicciones. Compare también el patrón de residuos y el condicionamiento. No interprete esta muestra como evidencia causal.

In [ ]:
modelos = {
    "Recta": ["constante", "lineal"],
    "Cuadrática": ["constante", "lineal", "cuadratica"],
    "Tendencia y ciclo": ["constante", "lineal", "seno", "coseno"],
    "Curvatura y ciclo": ["constante", "lineal", "cuadratica", "seno", "coseno"],
}
# ACTIVIDAD: añada aquí modelos["Mi propuesta"] = [claves del catálogo]
resultados = []
for nombre, claves in modelos.items():
    Am = matriz_diseno(x, claves)
    cm, _, _, _ = ajustar_qr(Am, y)
    rm = y-Am@cm
    resultados.append({"modelo": nombre, "p": len(claves),
                       "SSE": rm@rm, "RMSE": np.sqrt(np.mean(rm**2)),
                       "cond(A)": np.linalg.cond(Am),
                       "cond(A^T A)": np.linalg.cond(Am.T@Am)})
display(pd.DataFrame(resultados).set_index("modelo"))

**Respuesta 7.** Registre su quinta selección y sus resultados. Elija un modelo y justifique la elección con la gráfica, los residuos y la complejidad. Explique por qué agregar bases puede reducir el error sin mejorar una extrapolación.

## 7. Entrega y criterios de revisión
Entregue el cuaderno con celdas ejecutadas y respuestas completas. Debe incluir:
- datos, catálogo elegido y fórmula de la aproximación;
- filas verificadas de la matriz de diseño;
- ecuaciones normales con coeficientes numéricos;
- resolución QR y comprobaciones de reconstrucción, ortogonalidad y sistema triangular;
- tabla de valores ajustados, residuos y gráficos;
- comparación de cinco modelos y conclusión justificada.

| Criterio | Puntos |
|---|---:|
| Modelo, catálogo y evaluación de funciones | 4 |
| Formación e interpretación de las ecuaciones normales | 4 |
| Resolución con QR y verificación numérica | 6 |
| Residuos, comparación e interpretación | 6 |
| **Total** | **20** |

**Extensión opcional.** Agregue al catálogo una función `doble_lineal` igual a $2x$ y selecciónela junto con `lineal`. Examine el rango y explique por qué los coeficientes dejan de ser únicos. El solucionador debe detectar esta dependencia antes de realizar la sustitución.